In [36]:
import pandas as pd
import numpy as np

seasons =  pd.read_csv('original_data/seasons.csv')
circuits = pd.read_csv('original_data/circuits.csv')
drivers = pd.read_csv('original_data/drivers.csv')
constructors = pd.read_csv('original_data/constructors.csv')
races = pd.read_csv('original_data/races.csv')
race_results = pd.read_csv('original_data/results.csv')
sprint_results = pd.read_csv('original_data/sprint_results.csv')

In [39]:
collection = [seasons, circuits, drivers, constructors, races, race_results, sprint_results]

for dataframe in collection:
    dataframe.drop_duplicates(inplace=True)
    print(dataframe.isna().values.any())

AttributeError: 'DataFrame' object has no attribute 'Name'

In [14]:
seasons_clean = seasons.drop(['url'], axis=1)
seasons_clean.to_csv('clean_data/seasons.csv', index=False)

In [4]:
circuits_clean = circuits.drop(['circuitRef','lat','lng','alt','url'], axis=1)
circuits_clean.to_csv('clean_data/circuits.csv', index=False)

In [5]:
drivers_clean = drivers.drop(['driverRef','url'], axis=1)

drivers_clean.loc[drivers_clean['number'] == '\\N', 'number'] = 0
drivers_clean.loc[drivers_clean['code'] == '\\N', 'code'] = ""

drivers_clean.to_csv('clean_data/drivers.csv', index=False)

In [6]:
constructors_clean = constructors.drop(['constructorRef', 'url'], axis=1)
constructors_clean.to_csv('clean_data/constructors.csv', index=False)

In [7]:
races_clean = races.drop(['fp1_date','fp1_time','fp2_date','fp2_time','fp3_date','fp3_time','quali_date','quali_time','sprint_date','sprint_time', 'time'], axis=1)
races_clean.to_csv('clean_data/races.csv', index=False)

In [8]:
race_results_clean = race_results.drop(['resultId','positionText','positionOrder', 'laps','time','milliseconds','fastestLap','rank','fastestLapTime','fastestLapSpeed'],axis=1)
race_results_clean.to_csv('clean_data/results.csv',index=False)


In [11]:
driver_standings = pd.DataFrame(columns=["driverId", "position", "points","year"])

for current_year in range(1950, 2024):
    results_per_year = race_results_clean[['driverId', 'raceId', 'points']].merge(
        races_clean[['raceId', 'year']], on='raceId', how='left'
    )
    
    results_per_year = results_per_year[results_per_year['year'] == current_year]

    sprint_results_per_year = sprint_results[['driverId', 'raceId', 'points']].merge(
        races_clean[['raceId', 'year']], on='raceId', how='left'
    )
    sprint_results_per_year = sprint_results_per_year[sprint_results_per_year['year'] == current_year]
    
    combined_results = pd.concat([results_per_year, sprint_results_per_year], ignore_index=True)

    summed_values = combined_results.groupby('driverId', as_index=False)['points'].sum()

    sorted_values = summed_values.sort_values(by='points', ascending=False)
    sorted_values['position'] = range(1, len(sorted_values) + 1)
    sorted_values['year'] = current_year
    driver_standings = pd.concat([driver_standings, sorted_values[['driverId', 'position', 'points', 'year']]], ignore_index=True)

driver_standings.to_csv('clean_data/drivers_standings.csv')

C:\Users\JoaoCoutinho\AppData\Local\Temp\ipykernel_8012\4143355070.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  driver_standings = pd.concat([driver_standings, sorted_values[['driverId', 'position', 'points', 'year']]], ignore_index=True)


In [10]:
constructor_standings = pd.DataFrame(columns=["constructorId", "position", "points", "year"])

for current_year in range(1950, 2024):
    
    results_per_year = race_results_clean[['constructorId', 'raceId', 'points']].merge(
        races_clean[['raceId', 'year']], on='raceId', how='left'
    )

    results_per_year = results_per_year[results_per_year['year'] == current_year]

    sprint_results_per_year = sprint_results[['constructorId', 'raceId', 'points']].merge(
        races_clean[['raceId', 'year']], on='raceId', how='left'
    )
    sprint_results_per_year = sprint_results_per_year[sprint_results_per_year['year'] == current_year]

    combined_results = pd.concat([results_per_year, sprint_results_per_year], ignore_index=True)

    summed_values = combined_results.groupby('constructorId', as_index=False)['points'].sum()

    sorted_values = summed_values.sort_values(by='points', ascending=False)
    sorted_values['position'] = range(1, len(sorted_values) + 1)
    sorted_values['year'] = current_year


    constructor_standings = pd.concat([constructor_standings, sorted_values[['constructorId', 'position', 'points', 'year']]], ignore_index=True)
constructor_standings.to_csv('clean_data/constructors_standings.csv')


C:\Users\JoaoCoutinho\AppData\Local\Temp\ipykernel_8012\1617009900.py:25: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  constructor_standings = pd.concat([constructor_standings, sorted_values[['constructorId', 'position', 'points', 'year']]], ignore_index=True)
